In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..','..')))
import pandas as pd
from edbo.plus.optimizer_botorch import EDBOplus

#### This tutorial covers how to use **pre-computed PCA coordinates** as solvent descriptors in EDBO+, replacing the default One-Hot-Encoding (OHE).
#### Using PCA coordinates instead of OHE allows the Gaussian Process model to learn a **continuous, chemically meaningful** representation of solvent space, rather than treating each solvent as an independent binary category.
#### We will also show how to mix PCA-encoded solvents with other categorical parameters (e.g. base) that are still handled by OHE, and with standard numeric parameters (e.g. temperature, concentration).

## 1. Understanding the Solvent PCA Lookup Table.

##### EDBO+ ships with a lookup table of **272 common solvents** described by 5 principal components (PC1–PC5) derived from physicochemical solvent descriptors.
##### The columns are:
##### - $\bf{Name}$: Primary identifier — the solvent name used to look up entries.
##### - $\bf{CAS}$: Secondary identifier — CAS registry number.
##### - $\bf{PC1}$–$\bf{PC5}$: PCA score coordinates in the 5-dimensional solvent descriptor space.
##### Let's load and inspect the lookup table:

In [ ]:
solvent_lookup = pd.read_csv('../../DATA/Solvent_PC_clean.csv')
print(f"Lookup table contains {len(solvent_lookup)} solvents.")
solvent_lookup.head(10)

##### We can check how much variance each principal component explains across the 272 solvents.
##### This tells us how many PCs we actually need to include as features:

In [ ]:
pcs = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']
variances = solvent_lookup[pcs].var()
total_var = variances.sum()
explained = variances / total_var * 100
cumulative = explained.cumsum()

summary = pd.DataFrame({
    'Variance':      variances.round(3),
    'Explained (%)': explained.round(1),
    'Cumulative (%)': cumulative.round(1)
})
print(summary.to_string())
print()
print("=> PC1-PC4 capture 93.9% of solvent variance. PC5 adds only 6.1% and can be dropped.")

##### Based on this analysis, we will use **PC1–PC4** (93.9% of total variance) as solvent features throughout this tutorial.
##### You can search for any solvent by name to check its coordinates:

In [ ]:
# Search by keyword
solvent_lookup[solvent_lookup['Name'].str.contains('THF|DMSO|Toluene|Acetone|Methanol|DMF|DCM|Acetonitrile', case=False)]

## 2. Creating a Reaction Scope with PCA Solvent Encodings.

##### In the standard EDBO+ workflow, solvent names (strings) are automatically One-Hot-Encoded when the scope is loaded. This means each solvent becomes an independent binary column, and the model cannot exploit any chemical similarity between solvents.
##### EDBO+ now **auto-detects** solvent columns: when every value in a component column matches a name in the bundled lookup table, PC1–PC4 encodings are applied automatically — no extra configuration needed.
##### In this example we optimize a reaction with respect to **yield** and **enantioselectivity (ee)**, varying:
##### - $\bf{Solvent}$: 8 solvents — auto-encoded with PC1–PC4
##### - $\bf{Base}$: 3 categorical bases → One-Hot-Encoded as usual
##### - $\bf{Temperature}$: 4 numeric levels
##### - $\bf{Concentration}$: 3 numeric levels


In [ ]:
# Note: use exact names as they appear in the lookup table
reaction_components = {
    'solvent':       ['THF [Tetrahydrofuran]', 'Toluene', 'DMSO [Dimethylsulfoxide]',
                      'Acetone', 'Methanol', 'DMF [N,N-Dimethylformamide]',
                      'DCM [Dichloromethane]', 'Acetonitrile'],
    'base':          ['KOAc', 'Cs2CO3', 'K3PO4'],   # categorical → will be OHE'd
    'temperature':   [0, 25, 50, 80],                # numeric
    'concentration': [0.1, 0.2, 0.5],               # numeric
}


In [ ]:
# Solvents are auto-detected — no encodings dict needed.
# EDBO+ will print which columns were auto-encoded and remind you to use exclude_columns.
EDBOplus().generate_reaction_scope(
    components=reaction_components,
    filename='pca_optimization.csv',
    check_overwrite=False
)


##### Let's inspect the generated scope to confirm the PCA coordinates were joined correctly:

In [ ]:
df_scope = pd.read_csv('pca_optimization.csv')
print(f"Scope contains {len(df_scope)} reactions.")
print(f"Columns: {df_scope.columns.tolist()}")
print()
df_scope.head(10)

##### Notice that:
##### - The $\bf{solvent}$ column keeps the solvent name as a **readable label**.
##### - Four new numeric columns ($\bf{PC1}$–$\bf{PC4}$) encode the solvent properties — these are what the GP model will use.
##### - The $\bf{base}$ column is still a string — it will be One-Hot-Encoded automatically when we run EDBO+.
##### - $\bf{temperature}$ and $\bf{concentration}$ are numeric and will be used as-is.
##### This is a **296-reaction scope** (8 solvents × 3 bases × 4 temperatures × 3 concentrations). In each EDBO+ round we will run a small batch, making the search efficient.

## 3. First Steps: Initializing EDBO+ (No Training Data Yet).

##### Since we have not collected any experimental observations yet, EDBO+ will suggest an initial set of experiments using a space-filling sampling method (CVT sampling by default).
##### We pass $\bf{exclude\_columns=['solvent']}$ to tell EDBO+ that the $\bf{solvent}$ column is a **label-only** column — it should be kept in the CSV for readability, but should **not** be fed to the model (the PCA columns carry that information instead).
##### Without this, EDBO+ would see the solvent name as a categorical string and One-Hot-Encode it on top of the PC columns.

In [ ]:
EDBOplus().run(
    filename='pca_optimization.csv',          # Previously generated scope.
    objectives=['yield', 'ee'],               # Objectives to be optimized.
    objective_mode=['max', 'max'],            # Maximize both yield and ee.
    exclude_columns=['solvent'],              # Keep solvent name in CSV, but exclude from model.
    batch=4,                                  # Run 4 experiments per round.
    init_sampling_method='cvt',              # CVT space-filling initialization.
)

##### EDBO+ has added $\bf{yield}$, $\bf{ee}$ and $\bf{priority}$ columns to the CSV.
##### Notice the printout says $\bf{base}$ is One-Hot-Encoded, but **not** solvent — exactly as intended.
##### The top entries (with $\bf{priority=1}$) are the suggested first experiments:

In [ ]:
df_edbo = pd.read_csv('pca_optimization.csv')
print("Suggested initial experiments (priority = 1):")
df_edbo[df_edbo['priority'] == 1][['solvent', 'base', 'temperature', 'concentration', 'yield', 'ee', 'priority']]

## 4. Adding Training Data.

##### After running the suggested experiments in the lab, we enter the observed values into the CSV.
##### You can do this directly in any spreadsheet editor (Excel, LibreOffice Calc, etc.) or using Pandas as shown here.
##### We will simulate 4 experimental observations for demonstration purposes:

In [ ]:
df_edbo = pd.read_csv('pca_optimization.csv')

# Find the rows that were suggested (priority = 1)
suggested = df_edbo[df_edbo['priority'] == 1].index.tolist()
print("Indices of suggested experiments:", suggested)

# Simulated experimental observations
df_edbo.loc[suggested[0], 'yield'] = 32.1
df_edbo.loc[suggested[0], 'ee']    = 55.4

df_edbo.loc[suggested[1], 'yield'] = 67.8
df_edbo.loc[suggested[1], 'ee']    = 82.1

df_edbo.loc[suggested[2], 'yield'] = 18.5
df_edbo.loc[suggested[2], 'ee']    = 41.0

df_edbo.loc[suggested[3], 'yield'] = 51.3
df_edbo.loc[suggested[3], 'ee']    = 70.6

df_edbo.to_csv('pca_optimization_round0.csv', index=False)
print("Saved to pca_optimization_round0.csv")

##### Let's verify the data looks correct before running the optimizer:

In [ ]:
df_round0 = pd.read_csv('pca_optimization_round0.csv')

# Show observed rows
observed = df_round0[df_round0['yield'] != 'PENDING']
print(f"Observed experiments: {len(observed)}")
observed[['solvent', 'base', 'temperature', 'concentration', 'yield', 'ee']]

## 5. Running EDBO+ with Training Data.

##### Now that we have some experimental observations, we can run the full Bayesian optimization loop.
##### EDBO+ will:
##### 1. Fit a **Gaussian Process** surrogate model to the observed data, using PC1–PC4 as continuous solvent features and OHE-encoded base columns as categorical features.
##### 2. Optimize the **Noisy Expected Hypervolume Improvement (NoisyEHVI)** acquisition function across all untested reactions.
##### 3. Return the next batch of experiments ranked by $\bf{priority}$.

In [ ]:
EDBOplus().run(
    filename='pca_optimization_round0.csv',   # Scope including observations.
    objectives=['yield', 'ee'],               # Objectives to optimize.
    objective_mode=['max', 'max'],            # Maximize both.
    exclude_columns=['solvent'],              # Exclude solvent label; PC1-PC4 carry the information.
    batch=4,                                  # Suggest 4 experiments for the next round.
    init_sampling_method='cvt',
)

##### The top entries (with $\bf{priority=1}$) are the next recommended experiments.
##### Experiments already run have $\bf{priority=-1}$ and appear at the bottom.
##### Let's check the next suggested batch:

In [ ]:
df_round1 = pd.read_csv('pca_optimization_round0.csv')

print("Next suggested experiments (priority = 1):")
df_round1[df_round1['priority'] == 1][['solvent', 'base', 'temperature', 'concentration', 'yield', 'ee', 'priority']]

## 6. Accessing Model Predictions.

##### Each time EDBO+ runs with training data it generates a second CSV file prefixed with $\bf{pred\_}$.
##### This file contains the full scope with **model predictions** for every untested reaction:
##### - $\bf{\{objective\}\_predicted\_mean}$: the GP's predicted value for that objective.
##### - $\bf{\{objective\}\_predicted\_std\_dev}$: prediction uncertainty (standard deviation).
##### - $\bf{\{objective\}\_expected\_improvement}$: expected improvement over the current best.
##### High std_dev means the model is uncertain — those reactions could be surprising.

In [ ]:
df_preds = pd.read_csv('pred_pca_optimization_round0.csv')

# Show prediction columns available
pred_cols = [c for c in df_preds.columns if 'predicted' in c or 'expected' in c]
print("Prediction columns:", pred_cols)
print()

# Show top 10 reactions by predicted yield for untested reactions
untested = df_preds[df_preds['yield'] == 'PENDING'].copy()
untested['yield_predicted_mean'] = untested['yield_predicted_mean'].astype(float)
untested.sort_values('yield_predicted_mean', ascending=False)[
    ['solvent', 'base', 'temperature', 'concentration',
     'yield_predicted_mean', 'yield_predicted_std_dev', 'yield_expected_improvement',
     'ee_predicted_mean', 'ee_predicted_std_dev']
].head(10)

##### You can also visualize the prediction uncertainty using Pandas styling:

In [ ]:
display_cols = ['solvent', 'base', 'temperature', 'concentration', 'priority',
                'yield_predicted_mean', 'yield_predicted_std_dev',
                'ee_predicted_mean',    'ee_predicted_std_dev']

df_preds[display_cols].head(20).style.background_gradient(subset=['priority'], cmap='plasma')

## Summary: PCA Encoding vs One-Hot Encoding

## Summary: PCA Encoding vs One-Hot Encoding


| | **One-Hot Encoding (default)** | **PCA Encoding** |
|---|---|---|
| **Representation** | One binary column per solvent | 4 continuous PC coordinates |
| **Solvent space** | Each solvent is independent | Chemically similar solvents are close in PC space |
| **Scope size effect** | Adds N columns for N solvents | Always 4 columns, regardless of how many solvents |
| **Generalization** | Cannot interpolate between solvents | GP can interpolate across solvent space |
| **When to use** | Few solvents, no descriptor data | Many solvents, or when chemical similarity matters |

##### The minimal code pattern:

```python
# Scope generation — solvents auto-detected, no encodings dict needed
EDBOplus().generate_reaction_scope(
    components=reaction_components,
    filename='reaction.csv',
)

# Optimization — exclude the label column, let PC1-4 carry the information
EDBOplus().run(
    filename='reaction.csv',
    objectives=['yield', 'ee'],
    objective_mode=['max', 'max'],
    exclude_columns=['solvent'],
    batch=4,
)
```

##### To override auto-detection or use a custom feature set, pass an explicit `encodings` dict:

```python
EDBOplus().generate_reaction_scope(
    components=reaction_components,
    encodings={
        'solvent': {
            'file':     'data/Solvent_PC_clean.csv',
            'key':      'Name',
            'features': ['PC1', 'PC2', 'PC3', 'PC4'],
        }
    },
    filename='reaction.csv',
)
```
